In [1]:
"""
VECM Forecasting Model
=======================
Real-time recursive forecasts of log real TTF NG prices.
Specifications: VEC(1), VEC(12), VEC(AIC, p≤6).
Cointegrating rank r=1 imposed. Constant restricted to cointegrating
relation. All parameters re-estimated at each origin.
"""

import numpy as np
import pandas as pd
from statsmodels.tsa.vector_ar.vecm import VECM, select_order
import warnings
warnings.filterwarnings("ignore")

# ── Parameters ────────────────────────────────────────────────────────────────
HORIZONS     = [1, 3, 6, 9, 12, 15, 18, 21, 24]
EVAL_START   = "2015-01-01"
TRAIN_START  = "2006-02-01"
COINT_RANK   = 1
DET          = "ci"        # constant in cointegrating relation
AIC_LAG_MAX  = 6
INPUT_FILE   = "Input_VECM(Real_&_Log).xlsx"
OUTPUT_FILE  = "Output_VECM_forecasts.xlsx"

# Variable order: log_ng MUST be first (forecast target, normalised in beta)
VARS = ["log_ng", "log_oil", "log_coal"]

SPECIFICATIONS = [
    ("VEC(1)",        0,    False),   # (label, k_ar_diff, use_aic)
    ("VEC(12)",       11,   False),   # k_ar_diff = p-1 = 12-1 = 11
    ("VEC(AIC,p<=6)", None, True),
]

# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_excel(INPUT_FILE, sheet_name="Sheet1")
df.columns = ["date", "log_oil", "log_coal", "log_ng"]
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# Reorder columns so log_ng is first
df = df[["date", "log_ng", "log_oil", "log_coal"]]

df_train = df[df["date"] >= TRAIN_START].reset_index(drop=True)

# ── Helper: actual real price for a given year-month ─────────────────────────
def get_actual(ym_str):
    m = df[df["date"].dt.to_period("M").astype(str) == ym_str]
    if len(m) == 1:
        return np.exp(m["log_ng"].values[0])
    return np.nan

# ── Cleaner AIC selection using direct model comparison ──────────────────────
def select_aic_lag_direct(data, max_lag):
    """
    Fit VECM for each p=1..max_lag, compare AIC directly.
    Returns k_ar_diff = p-1.
    """
    best_aic = np.inf
    best_k   = 0
    for p in range(1, max_lag + 1):
        k = p - 1
        try:
            res = VECM(data, k_ar_diff=k, coint_rank=COINT_RANK,
                       deterministic=DET).fit()
            # statsmodels VECM does not expose .aic directly — use -2*llf + 2*nparams
            n_params = (k * data.shape[1]**2 +          # Gamma matrices
                        data.shape[1] * COINT_RANK * 2 + # alpha and beta
                        data.shape[1])                   # intercepts
            aic = -2 * res.llf + 2 * n_params
            if aic < best_aic:
                best_aic = aic
                best_k   = k
        except Exception:
            pass
    return best_k

# ── Main forecasting loop ─────────────────────────────────────────────────────
records      = []
eval_origins = df_train[df_train["date"] >= EVAL_START]["date"].tolist()

print(f"VECM forecasting: {len(eval_origins)} origins x "
      f"{len(SPECIFICATIONS)} specs x {len(HORIZONS)} horizons")
print(f"Specifications:  {[s[0] for s in SPECIFICATIONS]}")
print(f"Coint rank:      r={COINT_RANK}  (imposed)")
print(f"Deterministic:   '{DET}'  (constant in coint relation)")
print(f"AIC cap:         p<={AIC_LAG_MAX}  (Baumeister et al. 2024)")
print(f"Training:        {TRAIN_START} onwards")
print()

for i, origin_date in enumerate(eval_origins):

    history = df_train[df_train["date"] <= origin_date][VARS].values
    T       = len(history)

    if i % 20 == 0:
        print(f"  Origin {i+1}/{len(eval_origins)}: "
              f"{origin_date.strftime('%Y-%m-%d')}  T={T}")

    for label, fixed_k, use_aic in SPECIFICATIONS:

        # Determine k_ar_diff
        if use_aic:
            k = select_aic_lag_direct(history, AIC_LAG_MAX)
        else:
            k = fixed_k

        # Need at least k+2 observations (k lags + 1 difference + 1 level)
        if T < k + 3:
            continue

        try:
            res = VECM(history, k_ar_diff=k, coint_rank=COINT_RANK,
                       deterministic=DET).fit()
            # predict() returns array of shape (steps, K)
            # Column 0 = log_ng (forecast target)
            fc = res.predict(steps=max(HORIZONS))
        except Exception:
            continue

        for h in HORIZONS:
            actual_ym    = (origin_date + pd.DateOffset(months=h)).strftime("%Y-%m")
            forecast_lvl = np.exp(fc[h - 1, 0])
            actual_val   = get_actual(actual_ym)

            records.append({
                "forecast_origin": origin_date.strftime("%Y-%m-%d"),
                "horizon":         h,
                "model":           label,
                "actual_month":    actual_ym,
                "forecast":        forecast_lvl,
                "actual":          actual_val,
                "lag_order_used":  k + 1,    # report p (VAR order), not k_ar_diff
            })

# ── Save output ───────────────────────────────────────────────────────────────
results = pd.DataFrame(records)
results.to_excel(OUTPUT_FILE, index=False)

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("=" * 60)
print("VECM FORECASTING COMPLETE")
print("=" * 60)
print(f"  Total rows:       {len(results)}")
print(f"  Forecast origins: {results['forecast_origin'].nunique()}")
print(f"  Output:           {OUTPUT_FILE}")
print()

for label, _, _ in SPECIFICATIONS:
    sub      = results[results["model"] == label]
    lag_dist = (sub.drop_duplicates("forecast_origin")["lag_order_used"]
                   .value_counts().sort_index().to_dict())
    print(f"  {label}: {sub['forecast_origin'].nunique()} origins  "
          f"lag dist: {lag_dist}")

print()
first_origin = results["forecast_origin"].min()
sample = results[results["forecast_origin"] == first_origin]
print(f"Sample — first origin ({first_origin}):")
print(sample[["model","horizon","lag_order_used",
              "actual_month","forecast","actual"]].to_string(index=False))

VECM forecasting: 132 origins x 3 specs x 9 horizons
Specifications:  ['VEC(1)', 'VEC(12)', 'VEC(AIC,p<=6)']
Coint rank:      r=1  (imposed)
Deterministic:   'ci'  (constant in coint relation)
AIC cap:         p<=6  (Baumeister et al. 2024)
Training:        2006-02-01 onwards

  Origin 1/132: 2015-01-31  T=108
  Origin 21/132: 2016-09-30  T=128
  Origin 41/132: 2018-05-31  T=148
  Origin 61/132: 2020-01-31  T=168
  Origin 81/132: 2021-09-30  T=188
  Origin 101/132: 2023-05-31  T=208
  Origin 121/132: 2025-01-31  T=228

VECM FORECASTING COMPLETE
  Total rows:       3564
  Forecast origins: 132
  Output:           Output_VECM_forecasts.xlsx

  VEC(1): 132 origins  lag dist: {1: 132}
  VEC(12): 132 origins  lag dist: {12: 132}
  VEC(AIC,p<=6): 132 origins  lag dist: {2: 84, 3: 48}

Sample — first origin (2015-01-31):
        model  horizon  lag_order_used actual_month  forecast    actual
       VEC(1)        1               1      2015-02 17.495398 23.022297
       VEC(1)        3        

In [2]:
# ── VEC(AIC) lag distribution ─────────────────────────────────────────────────
aic_df = (results[results["model"] == "VEC(AIC,p<=6)"]
          .drop_duplicates("forecast_origin")
          .copy())
aic_df["forecast_origin"] = pd.to_datetime(aic_df["forecast_origin"])
aic_df = aic_df.sort_values("forecast_origin").reset_index(drop=True)

dist  = aic_df["lag_order_used"].value_counts().sort_index()
total = len(aic_df)

print("\nVEC(AIC) lag order distribution across forecast origins:")
print(f"{'Lag p':<8} {'Count':<8} {'Share (%)':<10}")
print("-" * 28)
for p, count in dist.items():
    print(f"p={p:<6} {count:<8} {count/total*100:.1f}%")
print(f"{'Total':<8} {total:<8} 100.0%")
print(f"\nMost frequent: p={dist.idxmax()}  "
      f"({dist.max()} origins, {dist.max()/total*100:.1f}%)")



VEC(AIC) lag order distribution across forecast origins:
Lag p    Count    Share (%) 
----------------------------
p=2      84       63.6%
p=3      48       36.4%
Total    132      100.0%

Most frequent: p=2  (84 origins, 63.6%)
